# Pelatihan Model & Evaluasi: LSTM
Di sini kita memuat data preprocessed dan melatih model LSTM raksasa.

In [5]:
import numpy as np
import pandas as pd
import time
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Bidirectional, Dropout, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load preprocessed data
data = np.load('processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
TIME_STEPS = X_train.shape[1]
input_shape = (TIME_STEPS, X_train.shape[2])

with open('rps_scaler.pkl', 'rb') as f:
    rps_scaler = pickle.load(f)


## Arsitektur LSTM

In [6]:
model = Sequential([
    LSTM(128, activation='tanh', return_sequences=True, input_shape=input_shape),
    Dropout(0.2),
    LSTM(64, activation='tanh'),
    Dense(2)
], name="LSTM_Imdoukh")
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

/opt/jupyter-env/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "LSTM_Imdoukh"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 30)             │         3,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            62 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,022 (15.71 KB)

 Trainable params: 4,022 (15.71 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
csv_logger = CSVLogger('training_progress_LSTM.csv', append=True)
EPOCHS = 200
BATCH_SIZE = 64

print(f"\n--- Training: LSTM ---")
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, csv_logger], verbose=1)



--- Training: LSTM ---
Epoch 1/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0618 - mae: 0.1806 - val_loss: 0.0550 - val_mae: 0.1711
Epoch 2/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0542 - mae: 0.1678 - val_loss: 0.0548 - val_mae: 0.1696
Epoch 3/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0539 - mae: 0.1675 - val_loss: 0.0543 - val_mae: 0.1688
Epoch 4/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0536 - mae: 0.1671 - val_loss: 0.0533 - val_mae: 0.1661
Epoch 5/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0533 - mae: 0.1665 - val_loss: 0.0524 - val_mae: 0.1641
Epoch 6/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0532 - mae: 0.1664 - val_loss: 0.0517 - val_mae: 0.1641
Epoch 7/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0532 - mae: 0.1666 - val_loss: 0.0518 - val_mae: 0.1629
Epoch 8/50
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0532 - mae: 0.1661 - val_loss: 0.0518 - val_mae: 0.1632


## Evaluasi Kecepatan & Akurasi

In [8]:
single_sample = X_test[0:1]

# Pemanasan prediksi
_ = model.predict(single_sample, verbose=0)

# Precise inference benchmark
t_arr = []
for _ in range(30):
    t_s = time.time()
    model.predict(single_sample, verbose=0)
    t_arr.append(time.time() - t_s)
avg_inference_ms = np.mean(t_arr) * 1000

# Prediksi seluruh data test
pred = model.predict(X_test, verbose=0)
np.save('pred_LSTM.npy', pred)

results = {
    'LSTM': {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R_Squared (R²)': r2_score(y_test, pred),
        'Speed (ms)': avg_inference_ms
    }
}
display(pd.DataFrame(results).T)


,MSE,RMSE,MAE,R_Squared (R²),Speed (ms)
LSTM,0.056196,0.237056,0.170582,0.416269,64.88475
